# Teste de dashboard HTML/CSS/JavaScript no ambiente Spark

Este notebook é exclusivamente diagnóstico. Ele **não consulta DB2, Hive, HDFS, arquivos nem dados de cliente**. Todos os valores exibidos são sintéticos.

Objetivos:

1. confirmar o kernel Python/Jupyter e as APIs de saída rica;
2. confirmar a sessão `%%spark`, PySpark e a JVM Java;
3. testar HTML e CSS inline, sem CDN e sem acesso à internet;
4. testar JavaScript direto, eventos, SVG, Canvas e `iframe srcdoc`;
5. testar, separadamente, a ponte opcional `%%spark -o` entre o Spark remoto e o kernel local;
6. indicar qual estratégia pode ser usada no dashboard do cliente.

Execute as células em ordem. A célula da ponte `%%spark -o` está marcada como opcional porque o suporte à opção depende da versão/configuração do SparkMagic. Uma falha apenas nessa célula é um resultado diagnóstico, não significa que HTML/CSS/JavaScript estejam indisponíveis.


In [ ]:
import html
import json
import platform
import sys
import traceback

TESTES_HTML_AMBIENTE = {}

def registrar_teste(nome, ok, detalhe=''):
    TESTES_HTML_AMBIENTE[nome] = {
        'status': 'OK' if ok else 'FALHA',
        'detalhe': str(detalhe),
    }

try:
    from IPython import get_ipython
    from IPython.display import HTML, Javascript, display
    registrar_teste('API_IPYTHON_DISPLAY', True, 'HTML, Javascript e display importados.')
except Exception as exc:
    registrar_teste('API_IPYTHON_DISPLAY', False, f'{type(exc).__name__}: {exc}')
    raise

ipython_atual = get_ipython()
cell_magics = set()
if ipython_atual is not None:
    cell_magics = set(ipython_atual.magics_manager.magics.get('cell', {}).keys())

registrar_teste('KERNEL_PYTHON', True, sys.version.split()[0])
registrar_teste('IPYTHON_ATIVO', ipython_atual is not None, type(ipython_atual).__name__ if ipython_atual else 'ausente')
registrar_teste('MAGIC_SPARK_REGISTRADA', 'spark' in cell_magics, sorted(cell_magics))

print('[LOCAL] Python:', sys.version.replace('\n', ' '))
print('[LOCAL] Plataforma:', platform.platform())
print('[LOCAL] IPython:', type(ipython_atual).__name__ if ipython_atual else None)
print('[LOCAL] %%spark registrada:', 'spark' in cell_magics)
print('[LOCAL] Cell magics:', ', '.join(sorted(cell_magics)))


## 1. Runtime remoto Spark e Java

As duas células seguintes usam somente objetos sintéticos em memória. O primeiro resultado esperado é um JSON entre `SPARK_RUNTIME_BEGIN/END`; o segundo deve mostrar seis linhas fictícias e um pequeno agregado.


In [ ]:
%%spark

import json
import platform
import sys

runtime_spark = {
    'python': sys.version.split()[0],
    'plataforma_driver': platform.platform(),
    'spark_version': spark.version,
    'spark_app_name': spark.sparkContext.appName,
    'spark_master': spark.sparkContext.master,
    'java_version': spark.sparkContext._jvm.java.lang.System.getProperty('java.version'),
    'java_vendor': spark.sparkContext._jvm.java.lang.System.getProperty('java.vendor'),
    'timezone_spark': spark.conf.get('spark.sql.session.timeZone'),
}

print('SPARK_RUNTIME_BEGIN')
print(json.dumps(runtime_spark, ensure_ascii=False, sort_keys=True))
print('SPARK_RUNTIME_END')


In [ ]:
%%spark

import json
from decimal import Decimal
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType, LongType, StringType, StructField, StructType

schema_dashboard_teste = StructType([
    StructField('CD_TEMA', LongType(), False),
    StructField('NM_TEMA', StringType(), False),
    StructField('TIPO', StringType(), False),
    StructField('QT_TRANSACOES', LongType(), False),
    StructField('VL_TOTAL', DecimalType(15, 2), False),
])

linhas_dashboard_teste = [
    (1, 'Moradia', 'Saída', 8, Decimal('2450.00')),
    (2, 'Alimentação', 'Saída', 14, Decimal('1380.50')),
    (3, 'Mobilidade', 'Saída', 6, Decimal('620.00')),
    (4, 'Saúde', 'Saída', 3, Decimal('410.20')),
    (5, 'Renda', 'Entrada', 2, Decimal('7200.00')),
    (6, 'Investimentos', 'Entrada', 1, Decimal('850.00')),
]

df_dashboard_teste = spark.createDataFrame(linhas_dashboard_teste, schema_dashboard_teste)
df_dashboard_resumo = (
    df_dashboard_teste
    .groupBy('TIPO')
    .agg(
        F.sum('QT_TRANSACOES').alias('QT_TRANSACOES'),
        F.sum('VL_TOTAL').alias('VL_TOTAL'),
    )
    .orderBy('TIPO')
)

assert df_dashboard_teste.count() == 6
assert df_dashboard_resumo.count() == 2

print('[SPARK] Dados 100% sintéticos:')
df_dashboard_teste.orderBy('CD_TEMA').show(truncate=False)
print('[SPARK] Agregado sintético:')
df_dashboard_resumo.show(truncate=False)

payload_spark_teste = [
    {
        'CD_TEMA': int(row['CD_TEMA']),
        'NM_TEMA': row['NM_TEMA'],
        'TIPO': row['TIPO'],
        'QT_TRANSACOES': int(row['QT_TRANSACOES']),
        'VL_TOTAL': str(row['VL_TOTAL']),
    }
    for row in df_dashboard_teste.orderBy('CD_TEMA').collect()
]
print('SPARK_PAYLOAD_SINTETICO_BEGIN')
print(json.dumps(payload_spark_teste, ensure_ascii=False))
print('SPARK_PAYLOAD_SINTETICO_END')


## 2. HTML e CSS no output do notebook

Se o teste funcionar, a próxima célula exibirá um dashboard estilizado, responsivo e sem dependências externas. Os números continuam sendo fictícios.


In [ ]:
dashboard_html_teste = r'''<div id="rf-test-dashboard" class="rf-shell">
  <style>
    #rf-test-dashboard, #rf-test-dashboard * { box-sizing: border-box; }
    #rf-test-dashboard {
      --ink: #14213d; --muted: #667085; --paper: #ffffff; --line: #e7eaf0;
      --blue: #2563eb; --cyan: #0891b2; --green: #059669; --orange: #ea580c;
      font-family: Inter, ui-sans-serif, system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
      color: var(--ink); background: linear-gradient(135deg, #f7faff 0%, #eef6ff 55%, #f8fafc 100%);
      border: 1px solid #dbe6f4; border-radius: 20px; padding: 24px; min-height: 520px;
    }
    #rf-test-dashboard .rf-head { display:flex; justify-content:space-between; gap:16px; align-items:flex-start; margin-bottom:22px; }
    #rf-test-dashboard h2 { margin:0; font-size:26px; letter-spacing:-0.03em; }
    #rf-test-dashboard .rf-sub { margin:6px 0 0; color:var(--muted); font-size:14px; }
    #rf-test-dashboard .rf-pill { color:#065f46; background:#d1fae5; border:1px solid #a7f3d0; border-radius:999px; padding:7px 11px; font-size:12px; font-weight:700; white-space:nowrap; }
    #rf-test-dashboard .rf-grid { display:grid; grid-template-columns:repeat(4,minmax(0,1fr)); gap:12px; }
    #rf-test-dashboard .rf-card { background:rgba(255,255,255,.94); border:1px solid var(--line); border-radius:15px; padding:16px; box-shadow:0 8px 24px rgba(15,23,42,.05); }
    #rf-test-dashboard .rf-label { color:var(--muted); font-size:12px; text-transform:uppercase; letter-spacing:.06em; font-weight:700; }
    #rf-test-dashboard .rf-value { display:block; margin-top:8px; font-size:25px; font-weight:800; letter-spacing:-.025em; }
    #rf-test-dashboard .rf-main { display:grid; grid-template-columns:1.35fr .65fr; gap:12px; margin-top:12px; }
    #rf-test-dashboard .rf-row { display:grid; grid-template-columns:110px 1fr 84px; gap:10px; align-items:center; margin:14px 0; font-size:13px; }
    #rf-test-dashboard .rf-track { height:10px; background:#eef2f6; border-radius:999px; overflow:hidden; }
    #rf-test-dashboard .rf-bar { height:100%; border-radius:999px; background:linear-gradient(90deg,var(--blue),var(--cyan)); }
    #rf-test-dashboard .rf-amount { text-align:right; font-variant-numeric:tabular-nums; font-weight:700; }
    #rf-test-dashboard .rf-ring { width:150px; height:150px; margin:18px auto 8px; border-radius:50%; background:conic-gradient(var(--green) 0 61%, #e6edf5 61% 100%); position:relative; }
    #rf-test-dashboard .rf-ring::after { content:"61%"; position:absolute; inset:17px; display:grid; place-items:center; background:white; border-radius:50%; font-size:28px; font-weight:800; }
    #rf-test-dashboard .rf-foot { color:var(--muted); font-size:11px; text-align:center; margin-top:10px; }
    @media (max-width: 900px) { #rf-test-dashboard .rf-grid { grid-template-columns:repeat(2,1fr); } #rf-test-dashboard .rf-main { grid-template-columns:1fr; } }
    @media (max-width: 520px) { #rf-test-dashboard { padding:15px; } #rf-test-dashboard .rf-grid { grid-template-columns:1fr; } #rf-test-dashboard .rf-row { grid-template-columns:90px 1fr 76px; } }
  </style>
  <div class="rf-head">
    <div><h2>Radar financeiro</h2><p class="rf-sub">Cliente sintético · ciclo de teste · nenhum dado real</p></div>
    <span class="rf-pill">HTML + CSS renderizados</span>
  </div>
  <section class="rf-grid" aria-label="Indicadores sintéticos">
    <article class="rf-card"><span class="rf-label">Entradas</span><strong class="rf-value">R$ 8.050</strong></article>
    <article class="rf-card"><span class="rf-label">Saídas</span><strong class="rf-value">R$ 4.861</strong></article>
    <article class="rf-card"><span class="rf-label">Saldo</span><strong class="rf-value" style="color:#047857">R$ 3.189</strong></article>
    <article class="rf-card"><span class="rf-label">Transações</span><strong class="rf-value">34</strong></article>
  </section>
  <section class="rf-main">
    <article class="rf-card">
      <span class="rf-label">Distribuição de saídas</span>
      <div class="rf-row"><span>Moradia</span><div class="rf-track"><div class="rf-bar" style="width:100%"></div></div><span class="rf-amount">R$ 2.450</span></div>
      <div class="rf-row"><span>Alimentação</span><div class="rf-track"><div class="rf-bar" style="width:56%"></div></div><span class="rf-amount">R$ 1.381</span></div>
      <div class="rf-row"><span>Mobilidade</span><div class="rf-track"><div class="rf-bar" style="width:25%"></div></div><span class="rf-amount">R$ 620</span></div>
      <div class="rf-row"><span>Saúde</span><div class="rf-track"><div class="rf-bar" style="width:17%"></div></div><span class="rf-amount">R$ 410</span></div>
    </article>
    <article class="rf-card"><span class="rf-label">Índice sintético</span><div class="rf-ring" role="img" aria-label="61 por cento"></div><div class="rf-foot">Teste de CSS conic-gradient</div></article>
  </section>
</div>'''

try:
    display(HTML(dashboard_html_teste))
    registrar_teste('HTML_CSS_INLINE', True, 'Objeto HTML enviado ao frontend.')
except Exception as exc:
    registrar_teste('HTML_CSS_INLINE', False, f'{type(exc).__name__}: {exc}')
    print(traceback.format_exc())


## 3. JavaScript, eventos e política de segurança

Há dois caminhos diferentes abaixo:

- um `<script>` embutido diretamente no HTML, que algumas políticas sanitizam;
- `IPython.display.Javascript`, que é o caminho nativo do Jupyter.

O teste está aprovado para interatividade se o selo **JavaScript via IPython: OK** aparecer e o botão responder. O script inline pode continuar bloqueado sem impedir um dashboard interativo.


In [ ]:
painel_js_teste = r'''<div id="rf-js-panel" style="font-family:system-ui;border:1px solid #dbe3ef;border-radius:14px;padding:16px;background:#fff;margin:8px 0">
  <div style="display:flex;gap:10px;flex-wrap:wrap;align-items:center">
    <span id="rf-inline-script-status" style="padding:6px 10px;border-radius:999px;background:#fff7ed;color:#9a3412;font-weight:700">Script inline: PENDENTE/BLOQUEADO</span>
    <span id="rf-ipython-js-status" style="padding:6px 10px;border-radius:999px;background:#fef2f2;color:#991b1b;font-weight:700">JavaScript via IPython: PENDENTE</span>
  </div>
  <button id="rf-js-button" type="button" style="margin-top:14px;border:0;border-radius:10px;padding:10px 14px;background:#1d4ed8;color:#fff;font-weight:700;cursor:pointer">Executar teste JS</button>
  <span id="rf-js-counter" style="margin-left:10px;color:#475569">Cliques: 0</span>
  <div id="rf-js-intl" style="margin-top:10px;color:#475569"></div>
  <script>
    (function(){
      var el = document.getElementById('rf-inline-script-status');
      if (el) { el.textContent = 'Script inline: OK'; el.style.background='#dcfce7'; el.style.color='#166534'; }
    })();
  </script>
</div>'''

display(HTML(painel_js_teste))
display(Javascript(r'''(function(){
  var status = document.getElementById('rf-ipython-js-status');
  var button = document.getElementById('rf-js-button');
  var counter = document.getElementById('rf-js-counter');
  var intl = document.getElementById('rf-js-intl');
  if (!status || !button || !counter || !intl) { return; }
  status.textContent = 'JavaScript via IPython: OK';
  status.style.background = '#dcfce7'; status.style.color = '#166534';
  intl.textContent = 'Intl.NumberFormat: ' + new Intl.NumberFormat('pt-BR', {style:'currency', currency:'BRL'}).format(3189.30);
  var clicks = 0;
  button.addEventListener('click', function(){ clicks += 1; counter.textContent = 'Cliques: ' + clicks + ' · evento OK'; });
})();'''))
registrar_teste('JAVASCRIPT_IPYTHON_ENVIADO', True, 'Confirmar visualmente selo e botão.')


In [ ]:
painel_graficos_teste = r'''<div id="rf-graphics-panel" style="font-family:system-ui;display:grid;grid-template-columns:repeat(auto-fit,minmax(240px,1fr));gap:12px;margin:8px 0">
  <div style="border:1px solid #dbe3ef;border-radius:14px;padding:16px;background:#fff">
    <strong>SVG inline</strong>
    <svg viewBox="0 0 320 120" role="img" aria-label="Gráfico SVG sintético" style="width:100%;margin-top:10px">
      <defs><linearGradient id="rf-svg-gradient" x1="0" x2="1"><stop stop-color="#2563eb"/><stop offset="1" stop-color="#06b6d4"/></linearGradient></defs>
      <polyline points="5,100 55,82 105,90 155,48 205,62 255,25 315,34" fill="none" stroke="url(#rf-svg-gradient)" stroke-width="6" stroke-linecap="round" stroke-linejoin="round"/>
      <circle cx="255" cy="25" r="7" fill="#059669"/>
    </svg>
    <span style="color:#166534;font-size:12px;font-weight:700">SVG: deve aparecer uma linha azul</span>
  </div>
  <div style="border:1px solid #dbe3ef;border-radius:14px;padding:16px;background:#fff">
    <strong>Canvas via JavaScript</strong>
    <canvas id="rf-canvas" width="320" height="120" style="width:100%;margin-top:10px;border-radius:8px;background:#f8fafc"></canvas>
    <span id="rf-canvas-status" style="color:#9a3412;font-size:12px;font-weight:700">Canvas: PENDENTE</span>
  </div>
</div>'''
display(HTML(painel_graficos_teste))
display(Javascript(r'''(function(){
  var canvas = document.getElementById('rf-canvas');
  var status = document.getElementById('rf-canvas-status');
  if (!canvas || !canvas.getContext) { if(status) status.textContent='Canvas: INDISPONÍVEL'; return; }
  var ctx = canvas.getContext('2d');
  var values = [42, 68, 51, 91, 74, 106];
  var colors = ['#bfdbfe','#93c5fd','#60a5fa','#3b82f6','#2563eb','#1d4ed8'];
  values.forEach(function(v,i){ ctx.fillStyle=colors[i]; ctx.fillRect(20+i*48,115-v,30,v); });
  status.textContent='Canvas via JavaScript: OK'; status.style.color='#166534';
})();'''))
registrar_teste('SVG_INLINE_ENVIADO', True, 'Confirmar linha SVG visualmente.')
registrar_teste('CANVAS_JAVASCRIPT_ENVIADO', True, 'Confirmar barras e selo visualmente.')


In [ ]:
iframe_srcdoc = r'''<!doctype html><html lang="pt-BR"><head><meta charset="utf-8"><style>
body{margin:0;padding:18px;font-family:system-ui;background:linear-gradient(135deg,#0f172a,#1e3a8a);color:white}
.card{border:1px solid rgba(255,255,255,.25);border-radius:14px;padding:16px;background:rgba(255,255,255,.1)}
#status{display:inline-block;margin-top:10px;padding:6px 10px;border-radius:999px;background:#78350f}
</style></head><body><div class="card"><strong>iframe srcdoc isolado</strong><div>CSS próprio e JavaScript sem CDN.</div><span id="status">JavaScript: PENDENTE</span></div>
<script>document.getElementById('status').textContent='JavaScript no iframe: OK';document.getElementById('status').style.background='#166534';</script></body></html>'''

iframe_html = (
    '<iframe title="Teste de iframe do dashboard" sandbox="allow-scripts" '
    'style="width:100%;height:180px;border:0;border-radius:14px;margin:8px 0" srcdoc="'
    + html.escape(iframe_srcdoc, quote=True)
    + '"></iframe>'
)
try:
    display(HTML(iframe_html))
    registrar_teste('IFRAME_SRCDOC_ENVIADO', True, 'Confirmar selo verde dentro do iframe.')
except Exception as exc:
    registrar_teste('IFRAME_SRCDOC_ENVIADO', False, f'{type(exc).__name__}: {exc}')


## 4. Renderização iniciada dentro de `%%spark`

Este teste identifica o que existe no processo remoto. Ele tenta, de forma protegida, `IPython.display.HTML` e `displayHTML` quando disponíveis. Não há consulta nem escrita. Se aparecer apenas texto/JSON, a arquitetura correta será renderizar no kernel local.


In [ ]:
%%spark

import json

resultado_render_remoto = {
    'ipython_html_importavel': False,
    'ipython_display_sem_erro': False,
    'displayHTML_disponivel': callable(globals().get('displayHTML')),
    'displayHTML_sem_erro': False,
}

html_remoto_teste = '<div style="padding:12px;border:2px solid #2563eb;border-radius:10px"><b>HTML iniciado no processo Spark remoto</b></div>'

try:
    from IPython.display import HTML as HTMLRemoto, display as display_remoto
    resultado_render_remoto['ipython_html_importavel'] = True
    display_remoto(HTMLRemoto(html_remoto_teste))
    resultado_render_remoto['ipython_display_sem_erro'] = True
except Exception as exc:
    resultado_render_remoto['ipython_display_erro'] = f'{type(exc).__name__}: {exc}'

if resultado_render_remoto['displayHTML_disponivel']:
    try:
        displayHTML(html_remoto_teste)
        resultado_render_remoto['displayHTML_sem_erro'] = True
    except Exception as exc:
        resultado_render_remoto['displayHTML_erro'] = f'{type(exc).__name__}: {exc}'

print('SPARK_RENDER_DIAGNOSTICO_BEGIN')
print(json.dumps(resultado_render_remoto, ensure_ascii=False, sort_keys=True))
print('SPARK_RENDER_DIAGNOSTICO_END')
print('[INTERPRETAÇÃO] Se o card azul não apareceu, use IPython.display.HTML no kernel local.')


## 5. Ponte opcional Spark → kernel local

A próxima célula testa a opção padrão do SparkMagic `-o`, limitada a seis linhas sintéticas. Se a versão corporativa não aceitar `-o` ou `-n`, registre a mensagem de erro e prossiga para a célula seguinte. Isso não invalida os testes visuais anteriores.

Não use essa ponte para trazer transações detalhadas. Em um dashboard real, transfira somente a linha final de 80 colunas ou agregados pequenos já calculados no Spark.


In [ ]:
%%spark -o df_dashboard_export -n 10

from decimal import Decimal
from pyspark.sql.types import DecimalType, LongType, StringType, StructField, StructType

schema_dashboard_export = StructType([
    StructField('ORDEM', LongType(), False),
    StructField('INDICADOR', StringType(), False),
    StructField('VALOR', DecimalType(15, 2), False),
])
df_dashboard_export = spark.createDataFrame([
    (1, 'Entradas', Decimal('8050.00')),
    (2, 'Saídas', Decimal('4860.70')),
    (3, 'Saldo', Decimal('3189.30')),
], schema_dashboard_export).orderBy('ORDEM')


In [ ]:
ponte_disponivel = 'df_dashboard_export' in globals()
detalhe_ponte = 'Variável local ausente.'

if ponte_disponivel:
    objeto_exportado = globals()['df_dashboard_export']
    detalhe_ponte = f'tipo local={type(objeto_exportado).__module__}.{type(objeto_exportado).__name__}'
    try:
        if hasattr(objeto_exportado, 'to_dict'):
            registros_exportados = objeto_exportado.to_dict(orient='records')
        elif isinstance(objeto_exportado, list):
            registros_exportados = objeto_exportado
        else:
            registros_exportados = []
        linhas_tabela = ''.join(
            '<tr><td style="padding:8px;border-bottom:1px solid #e5e7eb">'
            + html.escape(str(registro.get('INDICADOR', '')))
            + '</td><td style="padding:8px;text-align:right;border-bottom:1px solid #e5e7eb">R$ '
            + html.escape(str(registro.get('VALOR', '')))
            + '</td></tr>'
            for registro in registros_exportados
        )
        display(HTML(
            '<div style="font-family:system-ui;border:1px solid #a7f3d0;background:#f0fdf4;border-radius:14px;padding:16px">'
            '<strong style="color:#166534">Ponte %%spark -o: OK</strong>'
            '<table style="width:100%;margin-top:10px;border-collapse:collapse">' + linhas_tabela + '</table></div>'
        ))
    except Exception as exc:
        ponte_disponivel = False
        detalhe_ponte = f'{type(exc).__name__}: {exc}'

registrar_teste('PONTE_SPARK_PARA_LOCAL', ponte_disponivel, detalhe_ponte)
if not ponte_disponivel:
    display(HTML(
        '<div style="font-family:system-ui;border:1px solid #fed7aa;background:#fff7ed;border-radius:14px;padding:16px">'
        '<strong style="color:#9a3412">Ponte %%spark -o não confirmada</strong>'
        '<div style="margin-top:6px;color:#7c2d12">Continue usando o dashboard local com payload pequeno copiado/serializado por mecanismo homologado.</div>'
        '<code style="display:block;margin-top:8px">' + html.escape(detalhe_ponte) + '</code></div>'
    ))


## 6. Resumo local

O resumo abaixo cobre o que o kernel Python consegue provar. Os itens executados no navegador exigem também conferência visual dos selos e gráficos apresentados acima. Os dois outputs Spark devem conter seus marcadores `BEGIN/END`.


In [ ]:
linhas_resumo = []
for nome, resultado_teste in TESTES_HTML_AMBIENTE.items():
    cor = '#166534' if resultado_teste['status'] == 'OK' else '#991b1b'
    fundo = '#dcfce7' if resultado_teste['status'] == 'OK' else '#fee2e2'
    linhas_resumo.append(
        '<tr>'
        '<td style="padding:9px;border-bottom:1px solid #e5e7eb;font-weight:650">' + html.escape(nome) + '</td>'
        '<td style="padding:9px;border-bottom:1px solid #e5e7eb"><span style="padding:4px 8px;border-radius:999px;color:' + cor + ';background:' + fundo + ';font-weight:800">' + html.escape(resultado_teste['status']) + '</span></td>'
        '<td style="padding:9px;border-bottom:1px solid #e5e7eb;color:#475569">' + html.escape(resultado_teste['detalhe']) + '</td>'
        '</tr>'
    )

resumo_html = '''<div style="font-family:system-ui;border:1px solid #dbe3ef;border-radius:16px;padding:18px;background:white">
<h3 style="margin:0 0 12px">Resultado dos testes locais</h3>
<table style="width:100%;border-collapse:collapse"><thead><tr><th style="text-align:left;padding:9px">Teste</th><th style="text-align:left;padding:9px">Status</th><th style="text-align:left;padding:9px">Detalhe</th></tr></thead><tbody>''' + ''.join(linhas_resumo) + '''</tbody></table>
<div style="margin-top:15px;padding:12px;border-radius:12px;background:#eff6ff;color:#1e3a8a">
<strong>Checklist visual:</strong> dashboard estilizado; selo JavaScript via IPython verde; botão incrementando; linha SVG; barras Canvas; iframe com selo verde; dois JSONs Spark entre marcadores.
</div></div>'''
display(HTML(resumo_html))

print('TESTE_HTML_CONCLUIDO')
print(json.dumps(TESTES_HTML_AMBIENTE, ensure_ascii=False, indent=2))


## Interpretação para o dashboard real

- Se HTML/CSS e `IPython.display.Javascript` passaram, o notebook pode exibir um dashboard interativo autocontido, sem servidor web e sem CDN.
- Se `%%spark -o` passou, o fluxo recomendado é: Spark calcula → exporta somente agregados pequenos → kernel local gera o HTML.
- Se `%%spark -o` falhou, HTML ainda pode ser usado; será necessário escolher outro mecanismo corporativo já homologado para mover apenas o resultado agregado ao kernel local.
- Se JavaScript foi bloqueado, ainda é possível entregar um dashboard estático com HTML, CSS e SVG.
- O teste remoto com `displayHTML` é apenas diagnóstico. A renderização local é preferível porque Spark/Livy deve continuar responsável por cálculo, não por interface.

Nenhuma célula deste notebook deve ser adaptada para coletar transações detalhadas. Para o Radar Financeiro, a interface deve receber somente o resultado final ou agregados mínimos e não sensíveis.
